# Install packages

In [0]:
!pip install nltk

In [0]:
!pip install spotipy

In [0]:
!pip install wordcloud

# Import packages

In [0]:
# basic packages

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
from collections import Counter
import os

# NLP packages

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

from wordcloud import WordCloud
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

# spotify packages

import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

In [0]:
nltk.download('vader_lexicon')

In [0]:
nltk.download("stopwords")

In [0]:
nltk.download("wordnet")

# Spotity clients

In [0]:
# create a proyect and retrieval the credentials

# https://developer.spotify.com/documentation/web-api


In [0]:
# create a Spotify client

client_credentials_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager)

# Search artist

In [0]:
# search artist

artist_name = 'The killers'
artist_results = sp.search(q=artist_name, type='artist', limit=5)

In [0]:
# show posible results

for artist_name in artist_results['artists']['items']:
    print(artist_name['name'])

In [0]:
# select the first coincidence

artist_metadata = artist_results['artists']['items'][0]

In [0]:
artist_metadata['name']

In [0]:
# extract metadata

followers = artist_metadata['followers']['total']
genero = ",".join(artist_metadata['genres'])
popularidad = artist_metadata['popularity']

# Get albums

In [0]:
# get albums

artist_id_s = artist_results['artists']['items'][0]['id']
list_artist_albums = sp.artist_albums(artist_id_s, album_type='album')

In [0]:
def parse_release_date(x):
    if len(str(x)) == 4:
        return pd.to_datetime(f"{x}-01-01")
    else:
        return pd.to_datetime(x)

# Compound score

- -1 → sentimiento extremadamente negativo
- 0 → sentimiento neutral
- +1 → sentimiento extremadamente positivo

**Interpretación**

- ≥ 0.05 Positivo
- ≤ -0.05 Negativo
- Entre -0.05 y 0.05 Neutral

In [0]:
def get_sentiment_label(x):
    if x >= 0.05:
        return 'Positivo'
    elif x <= -0.05:
        return 'Negativo'
    else:
        return 'Neutral' 

In [0]:
# build album dataframe with extra attributes

sid = SentimentIntensityAnalyzer()

rows = []

for album in list_artist_albums['items']:

    sentiment_scores = sid.polarity_scores(album['name'])

    rows.append({
        'album_id': album['id'],
        'album_name': album['name'],
        'album_type': album['type'],
        'total_tracks': album['total_tracks'],
        'release_date': album['release_date'],
        'num_markets': len(album.get('available_markets', [])),
        'album_cover': album['images'][0]['url'] if album.get('images') else None,
        'sentiment_neg': sentiment_scores['neg'],
        'sentiment_neu': sentiment_scores['neu'],
        'sentiment_pos': sentiment_scores['pos'],
        'compound_score': sentiment_scores['compound'],
    })

df_all_albumns = pd.DataFrame(rows)

df_all_albumns['release_date'] = df_all_albumns['release_date'].apply(parse_release_date)
df_all_albumns['popularity'] = popularidad
df_all_albumns['followers'] = followers
df_all_albumns['genres'] = genero
df_all_albumns['artist_name'] = artist_metadata['name']
df_all_albumns['artist_id'] = artist_id_s

In [0]:
df_all_albumns['album_year'] = df_all_albumns.release_date.dt.year

In [0]:
df_all_albumns

In [0]:
df_all_albumns.compound_score.mean()

In [0]:
df_year = (
    df_all_albumns
    .groupby('album_year')
    .agg(
        compound_score=('compound_score', 'mean'),
        total_tracks=('total_tracks', 'sum')
    )
    .reset_index()
)
df_year['album_year'] = df_year['album_year'].astype(str)

In [0]:
fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(
    df_year['album_year'],
    df_year['compound_score'],
    marker='o'
)
ax1.set_xlabel('Año del álbum')
ax1.set_ylabel('Compound score')
ax1.axhline(0)

ax2 = ax1.twinx()
ax2.bar(
    df_year['album_year'],
    df_year['total_tracks'],
    alpha=0.3
)
ax2.set_ylabel('Total de tracks')

plt.title('Sentimiento (compound) y cantidad de tracks por año')
plt.show()

# Album-level extra analysis

In [0]:
# tracks per album

df_all_albumns.sort_values('total_tracks', ascending=False)[['album_name','album_year','total_tracks']].head(10)

In [0]:
# market reach per album

sns.barplot(data=df_all_albumns.sort_values('album_year'), x='album_year', y='num_markets')
plt.xticks(rotation=90)

# Song analysis

In [0]:
nombre_artista = 'The killers'
resultados_artista = sp.search(q=nombre_artista, type='artist', limit=1)

In [0]:
# extract track info: title, album, and track id for later enrichment

if resultados_artista and 'artists' in resultados_artista:
    artistas = resultados_artista['artists']['items']
    if artistas:
        id_artista = artistas[0]['id']

        albums_artista = sp.artist_albums(id_artista, album_type='album')

        datos_canciones = []

        for album in albums_artista['items']:
            nombre_album = album['name']
            canciones_album = sp.album_tracks(album['id'])

            for cancion in canciones_album['items']:
                titulo_cancion = cancion['name']
                puntuaciones_sentimiento = sid.polarity_scores(titulo_cancion)
                label_sentiment = get_sentiment_label(puntuaciones_sentimiento['compound'])

                datos_canciones.append({
                    'track_id': cancion['id'],
                    'title_name': titulo_cancion,
                    'title_length': len(titulo_cancion),
                    'title_word_count': len(titulo_cancion.split()),
                    'compound_score': puntuaciones_sentimiento['compound'],
                    'sentiment': label_sentiment,
                    'album': nombre_album
                })

        df_song = pd.DataFrame(datos_canciones)

# Enrich songs with popularity, explicit and duration

`sp.album_tracks` no devuelve estos campos. Llamamos a `sp.tracks` por lotes de 50.

In [0]:
# batched enrichment

track_ids = df_song['track_id'].dropna().unique().tolist()

extra = []

for i in range(0, len(track_ids), 50):
    batch = track_ids[i:i+50]
    res = sp.tracks(batch)
    for t in res['tracks']:
        if t is None:
            continue
        extra.append({
            'track_id': t['id'],
            'popularity': t['popularity'],
            'explicit': t['explicit'],
            'duration_min': t['duration_ms'] / 60000,
        })

df_extra = pd.DataFrame(extra)
df_song = df_song.merge(df_extra, on='track_id', how='left')
df_song.head()

In [0]:
df_song

In [0]:
# sort by compound score

df_song.sort_values('compound_score', ascending=False)

# Top songs by popularity

In [0]:
df_song.sort_values('popularity', ascending=False).head(10)

In [0]:
df_song.sort_values('popularity', ascending=True).head(10)

# Sentiment label distribution

In [0]:
df_song['sentiment'].value_counts()

In [0]:
sns.countplot(data=df_song, x='sentiment', order=['Negativo','Neutral','Positivo'])

In [0]:
# stacked distribution per album

df_stack = df_song.groupby(['album','sentiment']).size().unstack(fill_value=0)
df_stack = df_stack[[c for c in ['Negativo','Neutral','Positivo'] if c in df_stack.columns]]

df_stack.plot(kind='barh', stacked=True, figsize=(10, max(4, 0.35*len(df_stack))))
plt.xlabel('número de canciones')

# Sentiment vs popularity

¿Las canciones con título positivo son más populares?

In [0]:
df_song.groupby('sentiment').agg({
    'popularity': ['mean','median','count'],
    'compound_score': 'mean',
    'duration_min': 'mean',
})

In [0]:
sns.boxplot(data=df_song, x='sentiment', y='popularity', order=['Negativo','Neutral','Positivo'])

In [0]:
sns.scatterplot(data=df_song, x='compound_score', y='popularity', hue='explicit')

# Title length analysis

In [0]:
sns.scatterplot(data=df_song, x='title_word_count', y='popularity')

In [0]:
sns.scatterplot(data=df_song, x='title_length', y='compound_score')

# Album-level summary using enriched data

In [0]:
df_album_summary = df_song.groupby('album').agg(
    n_tracks=('track_id','nunique'),
    avg_popularity=('popularity','mean'),
    avg_duration=('duration_min','mean'),
    avg_compound=('compound_score','mean'),
    pct_explicit=('explicit', lambda x: x.mean()*100),
).reset_index().rename(columns={'album':'album_name'})

df_album_summary = df_album_summary.merge(
    df_all_albumns[['album_name','release_date','album_year']],
    on='album_name',
    how='left'
).sort_values('album_year')

df_album_summary

In [0]:
# popularity per album across time

plt.figure(figsize=(10,5))
sns.lineplot(data=df_album_summary, x='album_year', y='avg_popularity', marker='o')

In [0]:
# explicit % per album

plt.figure(figsize=(10,4))
sns.barplot(data=df_album_summary.sort_values('album_year'), x='album_year', y='pct_explicit')
plt.xticks(rotation=90)

# Sentiment vs album popularity

In [0]:
sns.scatterplot(
    data=df_album_summary,
    x='avg_compound',
    y='avg_popularity',
    size='n_tracks',
    sizes=(40, 300)
)
plt.axvline(0, color='gray', linestyle='--')

# Correlation heatmap

In [0]:
num_cols = ['popularity','duration_min','compound_score','title_length','title_word_count']

plt.figure(figsize=(7,5))
sns.heatmap(df_song[num_cols].corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')

# Sentiment by song summary

- sentiments aggregated by album (the original analysis)
- now merged with the enriched album dataframe

In [0]:
df_sentiments_by_songs = df_song.groupby('album').agg({'compound_score':'mean'}).reset_index().rename({'album':'album_name'}, axis=1)

In [0]:
df_sentiments_by_songs

In [0]:
df_summary_by_album = df_sentiments_by_songs.merge(df_all_albumns[['album_name','release_date','album_year']], on='album_name', how='left')

In [0]:
df_summary_by_album

In [0]:
# lineplot

df_summary_by_album.groupby('album_year').agg({'compound_score':'mean'}).plot()

In [0]:
# relational plot

g = sns.lmplot(
    data=df_summary_by_album,
    x="album_year",
    y="compound_score",
)

g.ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
g.ax.tick_params(axis='x', rotation=90)

# Word cloud

In [0]:
os.makedirs('wordclouds', exist_ok=True)

In [0]:
def generate_wordcloud(artist_name):

    resultados_artista = sp.search(q=artist_name, type="artist", limit=1)

    if resultados_artista["artists"]["items"]:
        id_artista = resultados_artista["artists"]["items"][0]["id"]

        albums = sp.artist_albums(id_artista, album_type="album")

        titulos_canciones = []
        for album in albums["items"]:
            tracks = sp.album_tracks(album["id"])
            titulos_canciones.extend([track["name"] for track in tracks["items"]])

        stop_words = set(stopwords.words("english"))

        titulos_procesados = [" ".join([word.lower() for word in titulo.split() if word.lower() not in stop_words])
                              for titulo in titulos_canciones]

        # tf-idf algorithm

        vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=50)
        X = vectorizer.fit_transform(titulos_procesados)

        palabras_importantes = vectorizer.get_feature_names_out()

        valores_importancia = X.sum(axis=0).tolist()[0]
        diccionario_palabras = dict(zip(palabras_importantes, valores_importancia))

        # word cloud

        plt.figure(figsize=(10, 6))
        nube_palabras = WordCloud(width=800, height=400, background_color="white", colormap="coolwarm")
        nube_palabras.generate_from_frequencies(diccionario_palabras)
        plt.imshow(nube_palabras, interpolation="bilinear")
        plt.axis("off")
        plt.savefig(f"./wordclouds/wordcloud_{artist_name}.png")
        plt.close()

    else:
        print("No se encontró al artista.")

In [0]:
generate_wordcloud("The killers")

In [0]:
generate_wordcloud("taylor swift")

In [0]:
generate_wordcloud("linkin park")

# Word frequency comparison

Alternativa cuantitativa al wordcloud: top palabras y bigramas como barras.

In [0]:
def get_top_terms(artist_name, top_n=15, ngram=(1,1)):

    res = sp.search(q=artist_name, type='artist', limit=1)
    aid = res['artists']['items'][0]['id']
    albums = sp.artist_albums(aid, album_type='album')

    titles = []
    for album in albums['items']:
        tracks = sp.album_tracks(album['id'])
        titles.extend([t['name'] for t in tracks['items']])

    stop_words = set(stopwords.words('english'))
    cleaned = [' '.join(w.lower() for w in t.split() if w.lower() not in stop_words and w.isalpha()) for t in titles]

    vec = TfidfVectorizer(ngram_range=ngram, max_features=top_n)
    X = vec.fit_transform(cleaned)

    scores = X.sum(axis=0).tolist()[0]
    return pd.DataFrame({'term': vec.get_feature_names_out(), 'score': scores}).sort_values('score', ascending=False)

In [0]:
# unigrams for The Killers

df_terms = get_top_terms('The killers', top_n=15)
sns.barplot(data=df_terms, x='score', y='term')

In [0]:
# bigrams for The Killers

df_bigrams = get_top_terms('The killers', top_n=15, ngram=(2,2))
sns.barplot(data=df_bigrams, x='score', y='term')

In [0]:
# side by side comparison

artists = ['The killers', 'taylor swift', 'linkin park']

fig, axes = plt.subplots(1, len(artists), figsize=(15, 6), sharex=False)

for ax, a in zip(axes, artists):
    df_t = get_top_terms(a, top_n=12)
    sns.barplot(data=df_t, x='score', y='term', ax=ax)
    ax.set_title(a)

plt.tight_layout()

In [0]:

# gato loco